In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
from nltk.corpus import stopwords
from nltk import ngrams
from nltk.tokenize import word_tokenize
import re
import string
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from sklearn.preprocessing  import LabelEncoder
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,LSTM,GRU,Dropout,SimpleRNN
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [2]:
df=pd.read_csv('combined_data.csv')

In [3]:
df.head()

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [4]:
df.shape

(83448, 2)

In [5]:
df.isnull().sum()

,0
label,0
text,0


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['text']=df['text'].str.lower()

In [8]:
df.head()

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [9]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))
df['text'] = df['text'].apply(remove_punctuation)

In [10]:
stop_words=set(stopwords.words('english'))

In [11]:
def remove_stop_words_list(token_list):
    if isinstance(token_list, list):
        return [word for word in token_list if word not in stop_words]
    return [word for word in word_tokenize(str(token_list)) if word not in stop_words]

# Handle both list and string cases to avoid TypeError
df['text'] = df['text'].apply(remove_stop_words_list)

# 3. Join tokens back into strings
df['text'] = df['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

display(df.head())

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get medircations online qnb ikud viagra...
2,0,computer connection cnn com wednesday escapenu...
3,1,university degree obtain prosperous future mon...
4,0,thanks answers guys know checked rsync manual ...


In [12]:
from collections import Counter

# Separate spam and ham
spam_words = ' '.join(df[df['label'] == 1]['text']).split()
ham_words = ' '.join(df[df['label'] == 0]['text']).split()

# Count frequencies
spam_common = Counter(spam_words).most_common(10)
ham_common = Counter(ham_words).most_common(10)

print('Top 10 words in Spam (Label 1):')
for word, count in spam_common:
    print(f'{word}: {count}')

print('\nTop 10 words in Ham (Label 0):')
for word, count in ham_common:
    print(f'{word}: {count}')

Top 10 words in Spam (Label 1):
escapenumber: 333240
escapelong: 188104
com: 29001
http: 27756
per: 26141
x: 23821
pills: 23128
escapenumbermg: 20541
price: 18723
company: 15872

Top 10 words in Ham (Label 0):
escapenumber: 799937
http: 54399
r: 53451
enron: 52856
c: 45222
org: 42557
com: 40681
escapelong: 39075
ect: 34743
help: 32661


In [13]:
def remove_urls(text):
  return re.sub(r'http\S+', '', text)

In [14]:
df['text']=df['text'].apply(remove_urls)

In [15]:
x=df['text']
y=df['label']

In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [17]:
max_features=10000
tokenizer=Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(x_train)

x_train_seq=tokenizer.texts_to_sequences(x_train)
x_test_seq=tokenizer.texts_to_sequences(x_test)

word_index=tokenizer.word_index
vocab_size=len(word_index)

In [18]:
vocab_size

283507

In [19]:
average_length = sum(map(len,x_train_seq))/ len(x_train_seq)


In [20]:
average_length

153.9705952844603

In [21]:
max_length=500
x_train_pad=pad_sequences(x_train_seq,maxlen=max_length,padding='post')
x_test_pad=pad_sequences(x_test_seq,maxlen=max_length,padding='post')

In [22]:
label=LabelEncoder()

In [23]:
import tensorflow as tf
tf.keras.backend.clear_session()

embedding_length=32
max_features=5000
max_length=500

In [24]:
model_rnn=Sequential()
model_rnn.add(Embedding(input_dim=max_features,output_dim=embedding_length,mask_zero=True))
model_rnn.add(SimpleRNN(64,return_sequences=False))
model_rnn.add(Dropout(0.2))
model_rnn.add(Dense(1,activation='sigmoid'))

In [25]:
model_rnn.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [26]:
model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
model_lstm=Sequential()
model_lstm.add(Embedding(input_dim=max_features,output_dim=embedding_length,mask_zero=True))
model_lstm.add(LSTM(64,return_sequences=False,use_cudnn=False))
model_lstm.add(Dropout(0.2))
model_lstm.add(Dense(1,activation='sigmoid'))

In [38]:
model_lstm.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [42]:
model_gru=Sequential()
model_gru.add(Embedding(input_dim=max_features,output_dim=embedding_length,mask_zero=True))
model_gru.add(GRU(64,return_sequences=False,use_cudnn=False))
model_gru.add(Dropout(0.2))
model_gru.add(Dense(1,activation='sigmoid'))

In [43]:
model_gru.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [31]:
model_gru.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
epochs=10
batch_size=64

In [33]:
# history_rnn=model_rnn.fit(x_train_pad,y_train,epochs=epochs,batch_size=batch_size,validation_data=(x_test_pad,y_test),verbose=1)

Epoch 1/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 46s 40ms/step - accuracy: 0.9481 - loss: 0.1392 - val_accuracy: 0.9781 - val_loss: 0.0710
Epoch 2/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9797 - loss: 0.0669 - val_accuracy: 0.9721 - val_loss: 0.0783
Epoch 3/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9843 - loss: 0.0515 - val_accuracy: 0.9775 - val_loss: 0.0711
Epoch 4/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9901 - loss: 0.0337 - val_accuracy: 0.9808 - val_loss: 0.0626
Epoch 5/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9931 - loss: 0.0254 - val_accuracy: 0.9578 - val_loss: 0.1628
Epoch 6/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9901 - loss: 0.0331 - val_accuracy: 0.9742 - val_loss: 0.0820
Epoch 7/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9950 - loss: 0.0174 - val_accuracy: 0.9769 - val_loss: 0.0913
Epoch 8/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9920 -

In [40]:
# Predict on the test data
y_pred_rnn = model_rnn.predict(x_test_pad)
y_pred_rnn_classes = (y_pred_rnn > 0.5).astype(int)

522/522 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step


### RNN Model Evaluation

In [41]:
# Confusion Matrix
print('Confusion Matrix (RNN):')
print(confusion_matrix(y_test, y_pred_rnn_classes))

# Classification Report
print('\nClassification Report (RNN):')
print(classification_report(y_test, y_pred_rnn_classes))

Confusion Matrix (RNN):
[[7735  203]
 [ 171 8581]]

Classification Report (RNN):
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      7938
           1       0.98      0.98      0.98      8752

    accuracy                           0.98     16690
   macro avg       0.98      0.98      0.98     16690
weighted avg       0.98      0.98      0.98     16690



In [39]:
# history_lstm=model_lstm.fit(x_train_pad,y_train,epochs=epochs,batch_size=batch_size,validation_data=(x_test_pad,y_test),verbose=1)

Epoch 1/10
 169/1044 ━━━━━━━━━━━━━━━━━━━━ 35s 41ms/step - accuracy: 0.8036 - loss: 0.3996

KeyboardInterrupt: 

In [44]:
history_gru=model_gru.fit(x_train_pad,y_train,epochs=epochs,batch_size=batch_size,validation_data=(x_test_pad,y_test),verbose=1)

Epoch 1/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 63s 55ms/step - accuracy: 0.9595 - loss: 0.1216 - val_accuracy: 0.9814 - val_loss: 0.0609
Epoch 2/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 54s 52ms/step - accuracy: 0.9763 - loss: 0.0812 - val_accuracy: 0.9815 - val_loss: 0.0590
Epoch 3/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 55s 52ms/step - accuracy: 0.9742 - loss: 0.0808 - val_accuracy: 0.9805 - val_loss: 0.0588
Epoch 4/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 54s 51ms/step - accuracy: 0.9877 - loss: 0.0415 - val_accuracy: 0.9847 - val_loss: 0.0464
Epoch 5/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 54s 52ms/step - accuracy: 0.9904 - loss: 0.0315 - val_accuracy: 0.9859 - val_loss: 0.0431
Epoch 6/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 54s 52ms/step - accuracy: 0.9923 - loss: 0.0263 - val_accuracy: 0.9854 - val_loss: 0.0435
Epoch 7/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 54s 52ms/step - accuracy: 0.9939 - loss: 0.0212 - val_accuracy: 0.9857 - val_loss: 0.0445
Epoch 8/10
1044/1044 ━━━━━━━━━━━━━━━━━━━━ 54s 52ms/step - accuracy: 0.9951 -

In [45]:
# Predict on the test data for GRU
y_pred_gru = model_gru.predict(x_test_pad)
y_pred_gru_classes = (y_pred_gru > 0.5).astype(int)

522/522 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step


### GRU Model Evaluation

In [46]:
# Confusion Matrix for GRU
print('Confusion Matrix (GRU):')
print(confusion_matrix(y_test, y_pred_gru_classes))

# Classification Report for GRU
print('\nClassification Report (GRU):')
print(classification_report(y_test, y_pred_gru_classes))

Confusion Matrix (GRU):
[[7774  164]
 [  95 8657]]

Classification Report (GRU):
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7938
           1       0.98      0.99      0.99      8752

    accuracy                           0.98     16690
   macro avg       0.98      0.98      0.98     16690
weighted avg       0.98      0.98      0.98     16690



In [47]:
model_gru.save('gru_model.keras')

In [49]:
import pickle

In [50]:
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)

In [51]:
config={
    'max_features':max_features,
    'max_length':max_length,
    'embedding_length':embedding_length
}

In [52]:
with open('config.pickle', 'wb') as f:
    pickle.dump(config, f)

In [55]:
label_mapping={0:'Ham',1:'Spam'}

In [56]:
with open('label_mapping.pickle', 'wb') as f:
    pickle.dump(label_mapping, f)